In [1]:
# Import the libraries and load the CSV into a pandas DataFrame:

In [2]:
import pandas as pd
import sqlite3

df = pd.read_csv('retail_sales_dataset.csv')

In [3]:
df.head()

,Transaction ID,Date,Customer ID,Gender,Age,Product Category,Quantity,Price per Unit,Total Amount
0,1,2023-11-24,CUST001,Male,34,Beauty,3,50,150
1,2,2023-02-27,CUST002,Female,26,Clothing,2,500,1000
2,3,2023-01-13,CUST003,Male,50,Electronics,1,30,30
3,4,2023-05-21,CUST004,Male,37,Clothing,1,500,500
4,5,2023-05-06,CUST005,Male,30,Beauty,2,50,100


In [4]:
df.shape

(1000, 9)

In [5]:
# Create an in-memory SQLite database connection and store the DataFrame as a table:

In [6]:
conn = sqlite3.connect(':memory:') # Creates an in-memory database
df.to_sql('retail_dataset', conn, index=False, if_exists='replace')


1000

In [7]:
#Run SQL queries using pandas.read_sql_query() and store the result in a new DataFrame:

In [8]:
sql_query = """
SELECT *
FROM retail_dataset
LIMIT 5;
"""
result_df = pd.read_sql_query(sql_query, conn)

# Display the result
print(result_df)


   Transaction ID        Date Customer ID  Gender  Age Product Category  \
0               1  2023-11-24     CUST001    Male   34           Beauty   
1               2  2023-02-27     CUST002  Female   26         Clothing   
2               3  2023-01-13     CUST003    Male   50      Electronics   
3               4  2023-05-21     CUST004    Male   37         Clothing   
4               5  2023-05-06     CUST005    Male   30           Beauty   

   Quantity  Price per Unit  Total Amount  
0         3              50           150  
1         2             500          1000  
2         1              30            30  
3         1             500           500  
4         2              50           100  


In [9]:
def print_res(query,con):
    res = pd.read_sql_query(query, con)
    print(res)

# Data Exploration & Cleaning

# Total Records

In [12]:
# Check total no of records
sq= """
Select count(*)
from retail_dataset;
"""
print_res(sq,conn)

   count(*)
0      1000


# Check Null Values

In [14]:
# Check if any null values present
sq= """
SELECT *
FROM retail_dataset
WHERE `Transaction ID` IS NULL OR Date IS NULL OR `Customer ID` IS NULL 
   OR Gender IS NULL OR Age IS NULL OR `Product Category` IS NULL
   OR Quantity IS NULL OR `Price per Unit` IS NULL OR `Total Amount` IS NULL;
"""
print_res(sq,conn)

Empty DataFrame
Columns: [Transaction ID, Date, Customer ID, Gender, Age, Product Category, Quantity, Price per Unit, Total Amount]
Index: []


In [15]:
# emoving Incomplete Records

In [16]:
# sq="""DELETE FROM retail_dataset
# WHERE `Transaction ID` IS NULL OR Date IS NULL OR `Customer ID` IS NULL 
#    OR Gender IS NULL OR Age IS NULL OR `Product Category` IS NULL
#    OR Quantity IS NULL OR `Price per Unit` IS NULL OR `Total Amount` IS NULL;"""
# print_res(sq,conn)

# Unique Customers

In [18]:
# Check Count of customers
sq = """SELECT COUNT(DISTINCT `Customer ID`) FROM retail_dataset;"""
print_res(sq,conn)

   COUNT(DISTINCT `Customer ID`)
0                           1000


# Unique Transactions

In [20]:
# Transaction ID is primary key
sq = """SELECT COUNT(DISTINCT `Transaction ID`) FROM retail_dataset;"""
print_res(sq,conn)

   COUNT(DISTINCT `Transaction ID`)
0                              1000


# Product Categories

In [22]:
# Categories
sq = """SELECT DISTINCT `Product Category` FROM retail_dataset;"""
print_res(sq,conn)

  Product Category
0           Beauty
1         Clothing
2      Electronics


# Date Range

In [24]:
sq = """
Select min(Date),max(Date)
from retail_dataset;
"""
print_res(sq,conn)

    min(Date)   max(Date)
0  2023-01-01  2024-01-01


In [25]:
# Contains 1 year of dataset from Jan,2023 to Jan,2024

# Business Analysis & SQL Queries

# Total Sales by Gender

In [28]:
# Total no of transaction per gender
sq = """
Select Gender,Count(*) as `Total Count`,sum(`Total Amount`)as `Total Sales`
from retail_dataset
group by Gender
order by `Total Sales` DESC;
"""
print_res(sq,conn)

   Gender  Total Count  Total Sales
0  Female          510       232840
1    Male          490       223160


# Total Sales by Category

In [30]:
# Total transaction per category
sq = """
Select `Product Category`,Count(*) as `Total Count`,sum(`Total Amount`) as `Total Sales`
from retail_dataset
group by `Product Category`
order by `Total Sales` DESC;
"""
print_res(sq,conn)

  Product Category  Total Count  Total Sales
0      Electronics          342       156905
1         Clothing          351       155580
2           Beauty          307       143515


# Top 5 Sales by Date

In [32]:
# Top 5 sales on which date
sq = """
Select Date,sum(`Total Amount`) as `Total Sales`
from retail_dataset
group by Date
order by `Total Sales` DESC
LIMIT 5;
"""
print_res(sq,conn)

         Date  Total Sales
0  2023-05-23         8455
1  2023-05-16         7260
2  2023-06-24         6220
3  2023-02-17         5890
4  2023-08-05         5205


In [33]:
# Product wise sales on date 23rd may,23
sq = """
Select `Product Category`, Sum(`Total Amount`) As `Total Sales`
from retail_dataset
where Date ='2023-05-23'
group by `Product Category`;
"""
print_res(sq,conn)

  Product Category  Total Sales
0           Beauty         2225
1         Clothing         1200
2      Electronics         5030


In [34]:
# Product wise sales on date 23rd may,23, select the category having total sales >5000
sq = """
Select `Product Category`, Sum(`Total Amount`) As `Total Sales`
from retail_dataset
where Date ='2023-05-23'
group by `Product Category`
having `Total Sales`> 5000;
"""
print_res(sq,conn)

  Product Category  Total Sales
0      Electronics         5030


# Sales Analysis by Age

In [36]:
# Min age and max age of the customer who bought the product
sq = """
Select min(Age) as `Min Age`,max(Age) as `Max Age`
From retail_dataset;
"""
print_res(sq,conn)

   Min Age  Max Age
0       18       64


In [37]:
# Average Age of Beauty Category Customers
sq = """
Select Avg(Age) as `Avg Age`
From retail_dataset
where `Product Category` = 'Beauty';
"""
print_res(sq,conn)

     Avg Age
0  40.371336


In [38]:
# High-Value Transactions (> 1000)
sq = """
Select count(*) 
From retail_dataset
where `Total Amount` > 1000;
"""
print_res(sq,conn)

   count(*)
0       153


In [39]:
# Transactions by Gender and Category
sq = """
Select `Product Category`, Gender, count(*) As `Total Transactions`
From retail_dataset
Group by 1,2;
"""
print_res(sq,conn)

  Product Category  Gender  Total Transactions
0           Beauty  Female                 166
1           Beauty    Male                 141
2         Clothing  Female                 174
3         Clothing    Male                 177
4      Electronics  Female                 170
5      Electronics    Male                 172


In [41]:
# Top 5 Customers by Total Sales
sq = """
Select `Customer ID`, Sum(`Total Amount`) As `Total Sales`
From retail_dataset
Group by 1
Order BY 2 DESC
LIMIT 5;
"""
print_res(sq,conn)

  Customer ID  Total Sales
0     CUST970         2000
1     CUST946         2000
2     CUST927         2000
3     CUST875         2000
4     CUST832         2000


In [42]:
#Unique Customers per Category
sq = """
Select `Product Category`, Count(Distinct `Customer ID`) As `Unique Customers`
From retail_dataset
Group by 1;
"""
print_res(sq,conn)

  Product Category  Unique Customers
0           Beauty               307
1         Clothing               351
2      Electronics               342


In [94]:
# Sales in each month of the year

In [80]:
sq = """SELECT strftime('%Y', Date) AS Year, STRFTIME('%m', Date) AS Month
FROM retail_dataset;"""
print_res(sq,conn)


     Year Month
0    2023    11
1    2023    02
2    2023    01
3    2023    05
4    2023    05
..    ...   ...
995  2023    05
996  2023    11
997  2023    10
998  2023    12
999  2023    04

[1000 rows x 2 columns]


In [86]:
sq = """SELECT strftime('%Y', Date) AS Year, STRFTIME('%m', Date) AS Month,Sum(`Total Amount`) As `Total Sales` 
FROM retail_dataset
Group by 1, 2;"""
print_res(sq,conn)


    Year Month  Total Sales
0   2023    01        35450
1   2023    02        44060
2   2023    03        28990
3   2023    04        33870
4   2023    05        53150
5   2023    06        36715
6   2023    07        35465
7   2023    08        36960
8   2023    09        23620
9   2023    10        46580
10  2023    11        34920
11  2023    12        44690
12  2024    01         1530


In [90]:
sq = """SELECT *
FROM retail_dataset
where strftime('%Y', Date) = '2024';"""
print_res(sq,conn)

   Transaction ID        Date Customer ID Gender  Age Product Category  \
0             211  2024-01-01     CUST211   Male   42           Beauty   
1             650  2024-01-01     CUST650   Male   55      Electronics   

   Quantity  Price per Unit  Total Amount  
0         3             500          1500  
1         1              30            30  


In [104]:
# Best-Selling Month in Each Year

In [130]:
sq = """SELECT strftime('%Y', Date) AS Year, STRFTIME('%m', Date) AS Month, Avg(`Total Amount`) As `Avg Sales` 
FROM retail_dataset
Group by 1, 2;"""
print_res(sq,conn)

    Year Month   Avg Sales
0   2023    01  466.447368
1   2023    02  518.352941
2   2023    03  397.123288
3   2023    04  393.837209
4   2023    05  506.190476
5   2023    06  476.818182
6   2023    07  492.569444
7   2023    08  393.191489
8   2023    09  363.384615
9   2023    10  485.208333
10  2023    11  447.692308
11  2023    12  491.098901
12  2024    01  765.000000


In [134]:
sq = """
SELECT year, month, avg_sale
FROM (
    SELECT strftime('%Y', Date) AS year,
           strftime('%m', Date) AS month,
           AVG(`Total Amount`) AS avg_sale,
           RANK() OVER (
               PARTITION BY strftime('%Y', Date)
               ORDER BY AVG(`Total Amount`) DESC
           ) AS sales_rank
    FROM retail_dataset
    GROUP BY strftime('%Y', Date), strftime('%m', Date)
) t
WHERE sales_rank = 1;
"""
print_res(sq,conn)

   year month    avg_sale
0  2023    02  518.352941
1  2024    01  765.000000


In [43]:
# Close the connection when you are finished:

In [44]:
conn.close()
